# HAM10000 — Generación Sintética

Genera imágenes sintéticas usando el embedding `<mel-skin>` aprendido en `HAM10000_textual_inversion.ipynb`.

**Prerequisito**: `mel_skin_embedding_final.pt` guardado por el notebook de Textual Inversion.

El notebook es **resumible**: si la sesión se corta, vuelve a correrlo y retoma
desde donde quedó — las imágenes ya guardadas en Drive no se regeneran.

| Método | Imágenes | Tiempo estimado T4 |
|---|---|---|
| Textual Inversion (desde ruido) | 4 500 | ~6–8 h |
| Img2Img (variaciones de reales) | 2 403 | ~5–6 h |

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    import diffusers
    print(f'diffusers {diffusers.__version__}')
except ImportError:
    import subprocess, sys
    print('Instalando dependencias...')
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q', '-U',
        'diffusers', 'transformers', 'accelerate', 'safetensors', 'huggingface_hub',
    ])
    import diffusers

import torch

def get_device():
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f'CUDA GPU: {name}  ({vram:.1f} GB)')
        return torch.device('cuda')
    if torch.backends.mps.is_available():
        print('Apple Silicon MPS')
        return torch.device('mps')
    print('Sin GPU — usando CPU')
    return torch.device('cpu')

DEVICE     = get_device()
DTYPE      = torch.float16 if DEVICE.type == 'cuda' else torch.float32
GEN_DEVICE = 'cpu' if DEVICE.type == 'mps' else str(DEVICE)
print(f'device={DEVICE}  dtype={DTYPE}  gen_device={GEN_DEVICE}')

In [ ]:
import json
import zipfile
from datetime import datetime, timezone
from pathlib import Path

if IN_COLAB:
    drive.mount('/content/drive')
    DRIVE_ROOT  = Path('/content/drive/MyDrive/ham10000-augmentation')
    ZIP_PATH    = DRIVE_ROOT / 'melanoma_train_for_colab.zip'
    IMAGES_DIR  = Path('/content/melanoma_images')
    MODEL_DIR   = DRIVE_ROOT / 'models'
    OUT_ROOT    = DRIVE_ROOT / 'synthetic'
else:
    PROJECT_ROOT = Path.cwd()
    ZIP_PATH     = PROJECT_ROOT / 'data/processed/melanoma_train_for_colab.zip'
    IMAGES_DIR   = PROJECT_ROOT / 'data/processed/melanoma_images_tmp'
    MODEL_DIR    = PROJECT_ROOT / 'data/processed/ti_models'
    OUT_ROOT     = PROJECT_ROOT / 'data/synthetic'

OUT_TI      = OUT_ROOT / 'textual_inversion'
OUT_IMG2IMG = OUT_ROOT / 'img2img'

for d in (IMAGES_DIR, OUT_TI, OUT_IMG2IMG):
    d.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), (
    f'ZIP no encontrado:\n  {ZIP_PATH}\n\n'
    'Ejecuta: python scripts/augmentation/prepare_for_colab.py'
)
ckpt_path = MODEL_DIR / 'mel_skin_embedding_final.pt'
assert ckpt_path.exists(), (
    f'Embedding no encontrado:\n  {ckpt_path}\n\n'
    'Ejecuta HAM10000_textual_inversion.ipynb primero.'
)
print(f'ZIP:       {ZIP_PATH}')
print(f'Embedding: {ckpt_path}')
print(f'Salida:    {OUT_ROOT}')

In [ ]:
if not any(IMAGES_DIR.glob('*.jpg')):
    print('Extrayendo imagenes del ZIP...')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        for m in zf.infolist():
            if m.filename.startswith('images/') and m.filename.endswith('.jpg'):
                data = zf.read(m.filename)
                (IMAGES_DIR / Path(m.filename).name).write_bytes(data)
    print('Listo')
else:
    print('Imagenes ya extraidas')

image_paths = sorted(IMAGES_DIR.glob('*.jpg'))
print(f'Imagenes fuente: {len(image_paths)}')

## Estado del progreso

Corre esta celda al inicio de cada sesión para ver cuánto queda.

In [ ]:
N_GENERATE = 4500
STRENGTHS  = [0.5, 0.6, 0.7]
N_I2I_TOTAL = len(image_paths) * len(STRENGTHS)

ti_done  = len(list(OUT_TI.glob('ti_*.jpg')))
i2i_done = len(list(OUT_IMG2IMG.glob('i2i_*.jpg')))

print('=== PROGRESO ACTUAL ===')
print(f'  TI      : {ti_done:>5} / {N_GENERATE}    ({100*ti_done/N_GENERATE:.1f}%)')
print(f'  Img2Img : {i2i_done:>5} / {N_I2I_TOTAL}   ({100*i2i_done/N_I2I_TOTAL:.1f}%)')
print()
if ti_done < N_GENERATE:
    print(f'  Pendiente TI      : {N_GENERATE - ti_done} imagenes')
else:
    print('  TI completo')
if i2i_done < N_I2I_TOTAL:
    print(f'  Pendiente Img2Img : {N_I2I_TOTAL - i2i_done} imagenes')
else:
    print('  Img2Img completo')

## Parte A — Generación con Textual Inversion

Retoma automáticamente desde la última imagen guardada.

In [ ]:
import gc
import random
from diffusers import StableDiffusionPipeline
from tqdm.auto import tqdm

MODEL_ID    = 'runwayml/stable-diffusion-v1-5'
PLACEHOLDER = '<mel-skin>'

BATCH_SIZE = 4 if DEVICE.type == 'cuda' else 1
STEPS      = 30
GUIDANCE   = 7.5
NEGATIVE   = 'low quality, blurry, cartoon, drawing, illustration, text, watermark'

PROMPTS = [
    f'dermoscopic image of a {PLACEHOLDER} skin lesion, high detail',
    f'dermatoscopy of {PLACEHOLDER} melanoma lesion, clinical',
    f'close-up dermoscopy of {PLACEHOLDER} pigmented lesion',
    f'medical dermoscopy, {PLACEHOLDER} irregular lesion, ABCD criteria',
    f'{PLACEHOLDER} skin lesion dermoscopy, asymmetric borders',
]

print(f'batch={BATCH_SIZE}  steps={STEPS}  guidance={GUIDANCE}')

In [ ]:
# Saltar si ya está completo
ti_done = len(list(OUT_TI.glob('ti_*.jpg')))
if ti_done >= N_GENERATE:
    print(f'TI ya completo ({ti_done} imagenes). Saltar a Parte B.')
else:
    print(f'Cargando pipeline (primera vez ~4 GB)...')
    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, safety_checker=None
    ).to(DEVICE)

    if PLACEHOLDER not in pipe.tokenizer.get_vocab():
        pipe.tokenizer.add_tokens([PLACEHOLDER])
    placeholder_id = pipe.tokenizer.convert_tokens_to_ids(PLACEHOLDER)
    pipe.text_encoder.resize_token_embeddings(len(pipe.tokenizer))

    ckpt = torch.load(ckpt_path, map_location='cpu')
    with torch.no_grad():
        pipe.text_encoder.get_input_embeddings().weight[placeholder_id] = ckpt['embedding'].to(DTYPE)

    pipe.enable_attention_slicing()
    print(f"Pipeline listo | '{PLACEHOLDER}' ID={placeholder_id}")
    print(f"Embedding: init='{ckpt.get('init_token','?')}'  steps={ckpt.get('step','?')}")

In [ ]:
ti_done = len(list(OUT_TI.glob('ti_*.jpg')))

if ti_done >= N_GENERATE:
    print(f'TI ya completo ({ti_done} imagenes). Continuar con Parte B.')
else:
    remaining = N_GENERATE - ti_done
    print(f'Retomando TI desde imagen {ti_done}  ({remaining} restantes)')

    generated = ti_done
    pbar = tqdm(total=remaining, desc='Generando TI')

    while generated < N_GENERATE:
        prompt  = random.choice(PROMPTS)
        batch_n = min(BATCH_SIZE, N_GENERATE - generated)
        seed    = random.randint(0, 2**32 - 1)
        gen     = torch.Generator(device=GEN_DEVICE).manual_seed(seed)

        images = pipe(
            prompt              = [prompt] * batch_n,
            negative_prompt     = [NEGATIVE] * batch_n,
            num_inference_steps = STEPS,
            guidance_scale      = GUIDANCE,
            generator           = gen,
            height=512, width=512,
        ).images

        for i, img in enumerate(images):
            img.save(OUT_TI / f'ti_{generated+i:05d}_s{seed}.jpg', quality=95)

        generated += batch_n
        pbar.update(batch_n)

    pbar.close()
    print(f'\nTI completo: {generated} imagenes -> {OUT_TI}')

## Parte B — Img2Img

Genera variaciones de las imágenes reales. Retoma automáticamente los pares
(imagen_fuente, strength) que ya están guardados.

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline
from PIL import Image

# Liberar VRAM del pipeline TI si estaba cargado
if 'pipe' in dir() and pipe is not None:
    del pipe
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
gc.collect()

PROMPT_I2I = 'dermoscopic image of a melanoma skin lesion, high detail, clinical'

i2i_done = len(list(OUT_IMG2IMG.glob('i2i_*.jpg')))
N_I2I_TOTAL = len(image_paths) * len(STRENGTHS)

if i2i_done >= N_I2I_TOTAL:
    print(f'Img2Img ya completo ({i2i_done} imagenes).')
    pipe_i2i = None
else:
    print(f'Img2Img: {i2i_done} hechas, {N_I2I_TOTAL - i2i_done} pendientes')
    print('Cargando pipeline img2img...')
    pipe_i2i = StableDiffusionImg2ImgPipeline.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, safety_checker=None
    ).to(DEVICE)
    pipe_i2i.enable_attention_slicing()
    print('Listo')

In [ ]:
if pipe_i2i is None:
    print('Img2Img ya completo, nada que hacer.')
else:
    generated_i2i = 0
    skipped_i2i   = 0
    pbar = tqdm(total=N_I2I_TOTAL, desc='Img2Img')

    for img_path in image_paths:
        init_img = None  # carga lazy: solo si hace falta alguna variacion

        for strength in STRENGTHS:
            # Nombre determinista por (fuente, strength); seed va en el sufijo
            pattern = f'i2i_{img_path.stem}_str{int(strength*10)}_*.jpg'
            if any(OUT_IMG2IMG.glob(pattern)):
                skipped_i2i += 1
                pbar.update(1)
                continue

            if init_img is None:
                init_img = Image.open(img_path).convert('RGB').resize((512, 512))

            seed = random.randint(0, 2**32 - 1)
            gen  = torch.Generator(device=GEN_DEVICE).manual_seed(seed)

            result = pipe_i2i(
                prompt              = PROMPT_I2I,
                negative_prompt     = NEGATIVE,
                image               = init_img,
                strength            = strength,
                num_inference_steps = STEPS,
                guidance_scale      = GUIDANCE,
                generator           = gen,
            ).images[0]

            fname = OUT_IMG2IMG / f'i2i_{img_path.stem}_str{int(strength*10)}_s{seed}.jpg'
            result.save(fname, quality=95)
            generated_i2i += 1
            pbar.update(1)

    pbar.close()
    print(f'\nImg2Img: {generated_i2i} nuevas generadas, {skipped_i2i} ya existian')
    print(f'Total en disco: {len(list(OUT_IMG2IMG.glob("i2i_*.jpg")))} -> {OUT_IMG2IMG}')

## Resumen y metadatos

In [ ]:
ti_count  = len(list(OUT_TI.glob('ti_*.jpg')))
i2i_count = len(list(OUT_IMG2IMG.glob('i2i_*.jpg')))

ckpt_meta = torch.load(ckpt_path, map_location='cpu')
meta = {
    'generated_at':      datetime.now(timezone.utc).isoformat(),
    'model':             MODEL_ID,
    'device':            str(DEVICE),
    'placeholder_token': PLACEHOLDER,
    'textual_inversion': {
        'init_token':      ckpt_meta.get('init_token', 'unknown'),
        'ti_steps':        ckpt_meta.get('step', 'unknown'),
        'n_generated':     ti_count,
        'inference_steps': STEPS,
        'guidance_scale':  GUIDANCE,
        'prompts':         PROMPTS,
        'negative':        NEGATIVE,
    },
    'img2img': {
        'prompt':          PROMPT_I2I,
        'negative':        NEGATIVE,
        'strengths':       STRENGTHS,
        'n_source_images': len(image_paths),
        'n_generated':     i2i_count,
        'inference_steps': STEPS,
        'guidance_scale':  GUIDANCE,
    },
    'total_synthetic': ti_count + i2i_count,
}

meta_path = OUT_ROOT / 'generation_metadata.json'
meta_path.write_text(json.dumps(meta, indent=2))

print('=== RESUMEN ===')
print(f'  Textual Inversion : {ti_count:>5} / {N_GENERATE}  ({100*ti_count/N_GENERATE:.1f}%)')
print(f'  Img2Img           : {i2i_count:>5} / {N_I2I_TOTAL} ({100*i2i_count/N_I2I_TOTAL:.1f}%)')
print(f'  Total sinteticas  : {ti_count + i2i_count:,}')
print(f'  Metadata          : {meta_path}')
if IN_COLAB:
    print('\nDescarga synthetic/ y generation_metadata.json de Google Drive a data/.')
else:
    print(f'\nImagenes en: {OUT_ROOT}')